================================================================================
模块 08：期权衍生品与波动率对冲 · 第 3/4 课
第 3 课：经典多腿期权组合策略与 Payoff 收益矩阵引擎 (Multi-Leg Option Strategies) - 参考答案版
================================================================================

【课程定位与投决会核心心智模型】
::: callout-note
### 📌 投决会核心心智模型：多腿期权是分段线性函数在希腊字母维度的空间矢量对消
单腿期权是对标的资产非线性凸性的基础表达；而多腿期权组合（Multi-Leg Structuring）则是机构投资者通过不同行权价（Strike）、不同到期日（Tenor）以及不同买卖方向（Long / Short）的代数叠加，在四维希腊字母空间中精确雕刻组合损益曲面（Payoff Profile）的高级工程。通过空间矢量互相对消，专业机构能够创造出‘Delta 中性且持续赚取时间租金（Theta > 0）’或‘下行损失绝对有界且上行爆发力充沛’的定制化资本结构。
:::

- **[到期分段线性的基底函数本质]**：任何复杂的到期损益函数，本质上都是标的现货 $S_T$、Call 基底 $\max(S_T - K, 0)$ 与 Put 基底 $\max(K - S_T, 0)$ 的线性组合。
- **[备兑看涨与保护性看跌的资本利差]**：Covered Call（持有股票 + 卖出 OTM Call）出让上方超额收益换取当期现金增厚（Yield Enhancement）；Protective Put（持有股票 + 买入 OTM Put）支付有限权利金为底仓资产购买刚性防灾保险。
- **[铁秃鹰的信用价差拓扑]**：Iron Condor 是对称的牛市看跌信用价差（Bull Put Spread）与熊市看涨信用价差（Bear Call Spread）的四腿闭环，在标的资产横盘震荡时最大化吞噬时间价值衰减。

【微观机理与三位一体就近图文剖析】

锚点指引：下图展示了四大经典期权组合（Covered Call / Protective Put / Straddle / Iron Condor）的几何到期收益拓扑对比。

```mermaid
flowchart TD
    subgraph 现货叠加型组合
        CC["备兑看涨 (Covered Call)<br/>现货 + Short Call ➔ 上行收益封顶 / 增厚分红"]
        PP["保护性看跌 (Protective Put)<br/>现货 + Long Put ➔ 下行刚性保底 / 上行无限"]
    end
    subgraph 纯衍生品多腿组合
        STR["买入跨式 (Long Straddle)<br/>Long ATM Call + Long ATM Put ➔ 纯做多波动率"]
        IC["铁秃鹰 (Iron Condor)<br/>四腿虚值牛熊组合 ➔ Delta中性 / 做空波动率收租"]
    end
```

- **[要点 1：备兑看涨的收益封顶代价]**：在牛市主升浪中，Covered Call 的股票将被动以行权价交割抛售，无法享受肥尾暴涨红利，长期复利跑输指数。
- **[要点 2：买入跨式的双刃剑属性]**：Long Straddle 两头通吃但必须支付双倍 Theta 租金，若标的价格未跳出双侧盈亏平衡点，组合将遭受时间价值与波动率崩塌（IV Crush）的惨烈双杀。
- **[要点 3：铁秃鹰的双边安全锁]**：相较于传统宽跨式空头（Short Strangle），铁秃鹰在最外侧配置了两腿深度虚值多头保护，将黑天鹅极端暴走的理论无限亏损严格限制为固定有界常数。

锚点指引：下图拆解了铁秃鹰（Iron Condor）四腿结构在中间平原区域的希腊字母空间对冲特征。

```mermaid
flowchart LR
    L1["Buy Put K1 (外侧深虚防灾)"] --> SpreadP["Bull Put Spread (净收权利金)"]
    L2["Sell Put K2 (内侧做空波动率)"] --> SpreadP
    L3["Sell Call K3 (内侧做空波动率)"] --> SpreadC["Bear Call Spread (净收权利金)"]
    L4["Buy Call K4 (外侧深虚防灾)"] --> SpreadC
    SpreadP & SpreadC --> Condor["铁秃鹰组合: Delta ≈ 0 | Theta > 0 | Vega < 0 | Max Loss 有界"]
```

- **[要点 1：中央平台的 Delta 抵消]**：做空 Put 的正 Delta 与做空 Call 的负 Delta 在行权价区间 $[K_2, K_3]$ 内相互抵消，组合对微小方向波动近乎免疫。
- **[要点 2：Theta 与 Vega 的卖方红利]**：内侧期权时间价值丰厚，外侧期权时间价值极薄，整个组合每日稳定收割时间价值损耗（Positive Theta）。
- **[要点 3：最大亏损精确可测]**：最大亏损严格等于翼宽（Wing Width, $K_2 - K_1$）减去期初净收取的权利金，保证金占用可控。

【数学与估值模型】
通用多腿组合到期净损益函数（Piecewise Linear Payoff）：
$$
\text{PnL}(S_T) = \sum_{i=1}^{M} w_i \cdot \text{Payoff}_i(S_T)
$$
单腿收益定义式：
- 标的股票：$\text{Payoff}_{\text{stock}}(S_T) = \text{side} \times (S_T - P_{\text{entry}})$
- 看涨期权：$\text{Payoff}_{\text{call}}(S_T) = \text{side} \times [\max(S_T - K, 0) - C_0]$
- 看跌期权：$\text{Payoff}_{\text{put}}(S_T) = \text{side} \times [\max(K - S_T, 0) - P_0]$
其中 $\text{side} \in \{+1, -1\}$（买入为 $+1$，卖出为 $-1$），$w_i \in \mathbb{N}^+$ 为合约张数。

铁秃鹰关键特征指标方程（$K_1 < K_2 < K_3 < K_4$）：
$$
\text{Net Credit} = (P_2 - P_1) + (C_3 - C_4), \quad \text{Max Profit} = \text{Net Credit}
$$
$$
\text{Max Loss} = (K_2 - K_1) - \text{Net Credit}, \quad \text{Breakevens} = \{K_2 - \text{Net Credit}, \; K_3 + \text{Net Credit}\}
$$
变量定义域：$0 < K_1 < K_2 < K_3 < K_4$，$\text{Net Credit} > 0$。

::: callout-warning
### ⚠️ 投决会风控红线与认知陷阱防范
1. **波动率崩塌（IV Crush）对买跨式的致命打击**：财报公布前夕买入 Straddle 成本极高（IV 处于 90% 极端分位）；财报落地后不确定性消除，IV 瞬间暴跌至 30%，Vega 巨亏直接淹没方向收益。
2. **铁秃鹰腿部被穿透时的流动性滑点**：当标的价格突破内侧行权价 $K_3$ 时，Gamma 激增使得亏损加速累积；若强平外侧保护腿，容易在流动性枯竭时遭遇恶劣滑点。
3. **备兑看涨误当无风险理财**：Covered Call 的权利金仅能提供 2%~3% 的下行缓冲；若遭遇个股基本面暴雷腰斩，底仓股票巨额浮亏将彻底吞噬多年的微薄权利金。
:::

【思考题】
Q1: 为什么在长周期回测中，长期机械性执行备兑看涨（Covered Call）策略在牛市主升浪中会严重跑输单纯持有股票指数（Buy-and-Hold）？
Q2: 铁秃鹰（Iron Condor）策略四腿组合在居中平值附近的希腊字母特征（Delta, Gamma, Vega, Theta）是什么？应在何种市场环境中部署？
Q3: 什么是“波动率崩塌（IV Crush）”？为什么很多散户在重大财报或大选公布前买入跨式（Long Straddle），即便标的股票剧烈波动依然遭遇严重亏损？

【Q1 深度参考答案】
- **[非对称收益分布的肥尾截断机制]**：备兑看涨策略的收益结构具有严重的负偏斜属性：它完整保留了标的资产全部下行风险（除微弱的权利金缓冲外），却将上方收益严格截断在行权价 $K$。
- **[牛市单边暴涨的复利缺口]**：标普500等指数历史上的长期复利绝大部分由极少数几个月或几周的极端暴涨贡献。机械执行 Covered Call 的基金在主升浪中被动交割出局，错失 30%~50% 的指数β收益；而在随后的熊市深跌中依然承受腰斩回撤，最终在几何复利累积上显著落后于被动持有策略。
- **[投决会动态增强方案]**：专业机构仅在隐含波动率极度高估（IV Rank > 80%）或宏观技术面确认严重超买阻力位时阶段性卖出虚值 Call，严禁进行全天候机械备兑。

【Q2 深度参考答案】
- **[居中平台希腊字母全景解析]**：
  1. **Delta 接近 0（Delta-Neutral）**：多腿虚值 Call 与 Put 在中间平台相互中和，对现货微小波动保持方向免疫。
  2. **Theta 显著为正（Positive Theta）**：内侧卖出的期权时间价值远比外侧买入的深虚期权丰厚，组合每天稳定吞噬时间价值租金。
  3. **Gamma 与 Vega 显著为负（Negative Gamma & Negative Vega）**：组合痛恨大单边跳空暴走与波动率突发膨胀。
- **[最佳部署环境约束]**：市场处于隐含波动率经历恐慌脉冲后的高位均值回归期（IV Percentile > 75%），且标的基本面进入催化剂真空期、预期价格在箱体内窄幅横盘。此时能够最大化收割膨胀的波动率风险溢价（VRP）与时间价值。

【Q3 深度参考答案】
- **[事件驱动型波动率风险溢价的物理本质]**：重大催化剂前夕（如英伟达财报发布前夜、美联储议息会议），全市场对冲需求使得双边期权隐含波动率被买盘推高至极端非理性高位（IV 从日常 35% 狂飙至 110%）。
- **[方差不确定性的瞬时清零与 IV Crush]**：业绩公告一旦公布，不论业绩是利好还是暴雷，悬顶的不确定性瞬间落地，市场避险买盘瞬间撤单，做市商光速下调期权报价，IV 发生垂直断崖式坍塌（如从 110% 断崖跌回 40%）。
- **[Vega 巨额亏损淹没 Delta 微利]**：Long Straddle 持有巨大的正 Vega（$\nu > 0$）。虽然股价发生了一定程度的跳空变动，但 IV 暴跌引发的 Vega 账面损耗（$-\nu \times \Delta \sigma$）以及跨式原本支付的高额时间损耗，往往远超跳空产生的内在收益，导致买方在波动发生时依然两头挨打、损失惨重。


In [ ]:
from typing import List, Dict, Any, Tuple


class MultiLegOptionEngine:
    """
    通用多腿期权与现货组合收益分析引擎。
    """

    @staticmethod
    def calculate_single_leg_payoff(leg: Dict[str, Any], spot_price: float) -> float:
        """
        计算单腿头寸在到期时点 spot_price 下的净收益 (PnL)。
        
        合法 leg 类型：
        - STOCK: {"type": "STOCK", "side": "BUY"|"SELL", "quantity": 1, "price": 100.0}
        - CALL:  {"type": "CALL", "side": "BUY"|"SELL", "quantity": 1, "strike": 105.0, "premium": 3.0}
        - PUT:   {"type": "PUT", "side": "BUY"|"SELL", "quantity": 1, "strike": 95.0, "premium": 2.0}
        """
        leg_type = leg["type"].upper()
        side_mult = 1.0 if leg.get("side", "BUY").upper() == "BUY" else -1.0
        qty = float(leg.get("quantity", 1))

        if leg_type == "STOCK":
            cost = float(leg.get("price", 0.0))
            return side_mult * (spot_price - cost) * qty

        elif leg_type == "CALL":
            strike = float(leg.get("strike", 0.0))
            premium = float(leg.get("premium", 0.0))
            intrinsic = max(0.0, spot_price - strike)
            return side_mult * (intrinsic - premium) * qty

        elif leg_type == "PUT":
            strike = float(leg.get("strike", 0.0))
            premium = float(leg.get("premium", 0.0))
            intrinsic = max(0.0, strike - spot_price)
            return side_mult * (intrinsic - premium) * qty

        return 0.0

    @classmethod
    def calculate_portfolio_payoff(cls, legs: List[Dict[str, Any]], spot_price: float) -> float:
        """计算整个多腿组合在到期价格 spot_price 下的总净收益。"""
        total = sum(cls.calculate_single_leg_payoff(leg, spot_price) for leg in legs)
        return round(total, 4)

    @classmethod
    def analyze_payoff_profile(
        cls,
        legs: List[Dict[str, Any]],
        spot_grid: List[float]
    ) -> Dict[str, Any]:
        """
        在给定的标的价格网格 spot_grid 上分析组合收益轮廓：
        - 最大收益 (max_profit)
        - 最大亏损 (max_loss, 负数或绝对亏损)
        - 盈亏平衡点列表 (breakeven_points)
        - 网格各点收益字典 (payoffs)
        """
        payoffs = [cls.calculate_portfolio_payoff(legs, s) for s in spot_grid]

        max_profit = max(payoffs)
        max_loss = min(payoffs)

        # 线性插值寻找零点 (盈亏平衡点)
        breakevens = []
        for i in range(len(spot_grid) - 1):
            s1, s2 = spot_grid[i], spot_grid[i + 1]
            p1, p2 = payoffs[i], payoffs[i + 1]

            if abs(p1) < 1e-4:
                breakevens.append(round(s1, 2))
            elif p1 * p2 < 0:
                root_s = s1 - p1 * (s2 - s1) / (p2 - p1)
                breakevens.append(round(root_s, 2))

        unique_bes = sorted(list(set(breakevens)))

        return {
            "max_profit": round(max_profit, 4),
            "max_loss": round(max_loss, 4),
            "breakeven_points": unique_bes,
            "min_spot": spot_grid[0],
            "max_spot": spot_grid[-1]
        }


# ==============================================================================
# 单元测试与断言校验
# ==============================================================================
if __name__ == "__main__":
    engine = MultiLegOptionEngine()

    print("[Testing] Covered Call Payoff Profile...")
    covered_call_legs = [
        {"type": "STOCK", "side": "BUY", "quantity": 1, "price": 100.0},
        {"type": "CALL", "side": "SELL", "quantity": 1, "strike": 105.0, "premium": 3.0}
    ]
    pnl_110 = engine.calculate_portfolio_payoff(covered_call_legs, 110.0)
    assert pnl_110 == 8.0, f"Expected 8.0, got {pnl_110}"

    pnl_90 = engine.calculate_portfolio_payoff(covered_call_legs, 90.0)
    assert pnl_90 == -7.0, f"Expected -7.0, got {pnl_90}"

    spot_range_cc = [float(x) for x in range(80, 130)]
    cc_profile = engine.analyze_payoff_profile(covered_call_legs, spot_range_cc)
    assert cc_profile["max_profit"] == 8.0
    assert 97.0 in cc_profile["breakeven_points"]

    print("[Testing] Protective Put Payoff Profile...")
    protective_put_legs = [
        {"type": "STOCK", "side": "BUY", "quantity": 1, "price": 100.0},
        {"type": "PUT", "side": "BUY", "quantity": 1, "strike": 95.0, "premium": 2.0}
    ]
    pnl_pp_80 = engine.calculate_portfolio_payoff(protective_put_legs, 80.0)
    assert pnl_pp_80 == -7.0

    pnl_pp_120 = engine.calculate_portfolio_payoff(protective_put_legs, 120.0)
    assert pnl_pp_120 == 18.0

    pp_profile = engine.analyze_payoff_profile(protective_put_legs, spot_range_cc)
    assert 102.0 in pp_profile["breakeven_points"]

    print("[Testing] Long Straddle Payoff Profile...")
    straddle_legs = [
        {"type": "CALL", "side": "BUY", "quantity": 1, "strike": 100.0, "premium": 4.0},
        {"type": "PUT", "side": "BUY", "quantity": 1, "strike": 100.0, "premium": 3.0}
    ]
    pnl_straddle_100 = engine.calculate_portfolio_payoff(straddle_legs, 100.0)
    assert pnl_straddle_100 == -7.0

    pnl_straddle_115 = engine.calculate_portfolio_payoff(straddle_legs, 115.0)
    assert pnl_straddle_115 == 8.0

    straddle_profile = engine.analyze_payoff_profile(straddle_legs, spot_range_cc)
    assert straddle_profile["max_loss"] == -7.0
    assert 93.0 in straddle_profile["breakeven_points"]
    assert 107.0 in straddle_profile["breakeven_points"]

    print("[Testing] Iron Condor Payoff Profile...")
    iron_condor_legs = [
        {"type": "PUT", "side": "BUY", "quantity": 1, "strike": 85.0, "premium": 1.0},
        {"type": "PUT", "side": "SELL", "quantity": 1, "strike": 90.0, "premium": 2.5},
        {"type": "CALL", "side": "SELL", "quantity": 1, "strike": 110.0, "premium": 2.5},
        {"type": "CALL", "side": "BUY", "quantity": 1, "strike": 115.0, "premium": 1.0}
    ]
    assert engine.calculate_portfolio_payoff(iron_condor_legs, 100.0) == 3.0
    assert engine.calculate_portfolio_payoff(iron_condor_legs, 75.0) == -2.0
    assert engine.calculate_portfolio_payoff(iron_condor_legs, 125.0) == -2.0

    spot_range_ic = [float(x) for x in range(70, 130)]
    ic_profile = engine.analyze_payoff_profile(iron_condor_legs, spot_range_ic)
    assert ic_profile["max_profit"] == 3.0
    assert ic_profile["max_loss"] == -2.0
    assert 87.0 in ic_profile["breakeven_points"]
    assert 113.0 in ic_profile["breakeven_points"]

    print("All assertions passed! Module 08 Lesson 03 verification successful.")
